# Notebook 06 — Discussion and Conclusions

**Module:** 18 — Scientific Writing and Open Science  
**Tier:** 2 — Working competence  
**Notebook:** 6 of 12  
**Time estimate:** 60 minutes

> The discussion is where you tell the reader what your results *mean*.
> It is the hardest section to write — and the section where overclaiming,
> vagueness, and circular argument are most dangerous.

---
## Step 1 — Motivation

Discussions fail in two opposite directions:
1. **Overclaiming:** "Our results demonstrate that deep learning solves the TF binding
   prediction problem." (No. One benchmark on one dataset does not.)
2. **Underdiscussing:** "We showed AUROC 0.924. Future work could explore more datasets."
   (This is not a discussion — it is a conclusion stub.)

The correct discussion interprets results in the context of prior work, explains
unexpected findings, acknowledges limitations honestly, and points to the specific
open questions that the data now makes addressable.

---
## Step 2 — Discussion Paragraph Structure

| Paragraph | Content | Common mistake |
|-----------|---------|---------------|
| **D1: Core interpretation** | What your key result means, in context of the field | Restating the results number without interpreting it |
| **D2: Comparison to prior work** | How your result compares to the 3–5 most relevant papers | Ignoring papers that disagree with you |
| **D3: Mechanism / explanation** | Why the result makes biological or algorithmic sense | Speculation without flagging it as speculation |
| **D4: Limitations** | What the method can't do; when it would fail | Omitting limitations to appear stronger |
| **D5: Future work** | Specific, testable next steps — not generic "more research needed" | "Future work could explore many directions" |

---
## Step 3 — Overclaiming vs. Appropriate Hedging

**Overclaiming** = claiming more than the data support.
**Underclaiming** = refusing to interpret anything.
**Appropriate hedging** = scoping claims precisely to the evidence.

| Overclaim | Appropriate version |
|-----------|--------------------|
| "Our model *proves* that cooperativity drives TF binding" | "Our model's performance gain is *consistent with* the hypothesis that cooperativity contributes to binding specificity" |
| "Deep learning *solves* TF binding prediction" | "Deep learning reduces the AUROC gap by 12 pp on the ENCODE benchmark" |
| "This will transform bioinformatics" | "This approach scales to genome-wide prediction without manual curation" |

**Hedging verbs:** suggest, indicate, are consistent with, support the hypothesis,
are compatible with. Avoid: prove, demonstrate definitively, conclusively show.

---
## Step 4 — The Limitation Paragraph

State limitations *before* reviewers do. A paper that anticipates its own weaknesses
reads as more trustworthy than one that pretends not to have any.

A good limitation sentence has two parts: the limitation and its implication.

- Weak: "Our study has some limitations."
- Strong: "Our model was trained exclusively on human ChIP-seq data; performance
  on non-human or non-ChIP assays (e.g., SELEX, CUT&RUN) remains untested and may
  differ systematically due to assay-specific biases."

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ---- Discussion auditing tools ----

OVERCLAIM_WORDS = [
    'prove', 'proves', 'proven', 'definitive', 'definitively', 'conclusive',
    'conclusively', 'revolutionary', 'transform', 'solves', 'solve', 'complete',
    'fully', 'always', 'never', 'all cases', 'universal'
]

HEDGE_WORDS = [
    'suggest', 'indicate', 'consistent with', 'support', 'compatible with',
    'may', 'might', 'could', 'appear', 'seem', 'likely', 'potentially'
]

LIMITATION_SIGNALS = [
    'limitation', 'caveat', 'caution', 'note that', 'important to note',
    'should be noted', 'restricted to', 'only', 'future work',
    'not tested', 'untested', 'may not generalize', 'does not account for'
]

FUTURE_WORK_GENERIC = [
    'future work could explore', 'further research is needed', 'more work is needed',
    'this is an interesting direction', 'could be investigated in the future'
]

def audit_discussion(text):
    t_lower = text.lower()
    overclaims = [w for w in OVERCLAIM_WORDS if w in t_lower]
    hedges = sum(1 for h in HEDGE_WORDS if h in t_lower)
    limitations = sum(1 for l in LIMITATION_SIGNALS if l in t_lower)
    future_generic = [f for f in FUTURE_WORK_GENERIC if f in t_lower]
    return {
        'overclaims': overclaims,
        'hedge_count': hedges,
        'limitation_signals': limitations,
        'generic_future_work': future_generic,
    }

DISC_WEAK = """
Our results prove that deep learning is the definitive solution to transcription factor
binding prediction. Our model achieves AUROC 0.924, which is better than all existing methods.
This will revolutionize computational biology and transform how scientists study gene regulation.
Our approach solves the long-standing problem of sequence-specific TF binding prediction.
Future work could explore more datasets. This is an interesting direction for future research.
"""

DISC_STRONG = """
DeepBind-v2's 12.3 percentage-point improvement over the PWM baseline is consistent with
the hypothesis that positional dependencies in TF binding sequences carry information that
independent-site models cannot capture. This interpretation aligns with structural data
showing cooperative contacts between TF domains and adjacent nucleotides in 14 of the 20
TFs for which crystal structures are available in the PDB.

Our findings complement those of Alipanahi et al. (2015), who showed that convolution-based
architectures can recover known TF binding motifs from DeepBind's filters. DeepBind-v2
extends this by adding batch normalization and dropout, which appear to improve both
generalization (3.1 pp gain on held-out TFs) and training stability.

We note two important limitations. First, our training data are exclusively from ChIP-seq
assays in human cell lines; performance on assays with different background distributions
(SELEX, CUT&RUN) or non-human organisms remains untested. Second, the model is trained
on 200 bp windows, and may not capture distal co-operative interactions at longer ranges.
Systems where long-range cooperativity is documented — notably pioneer TFs — may benefit
from architectures with larger receptive fields.
"""

for name, disc in [('WEAK', DISC_WEAK), ('STRONG', DISC_STRONG)]:
    result = audit_discussion(disc)
    print(f'\n=== {name} discussion ===')
    print(f'  Overclaim words: {result["overclaims"]}')
    print(f'  Hedge count: {result["hedge_count"]}')
    print(f'  Limitation signals: {result["limitation_signals"]}')
    print(f'  Generic future-work phrases: {result["generic_future_work"]}')

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 7))

# Panel A: Overclaim vs hedge audit
ax = axes[0]
criteria = ['Overclaim\nwords (0=good)', 'Hedge\nwords (≥3=good)',
             'Limitation\nsignals (≥2=good)', 'Generic\nfuture work (0=good)']

weak_r   = audit_discussion(DISC_WEAK)
strong_r = audit_discussion(DISC_STRONG)

weak_vals   = [len(weak_r['overclaims']),   weak_r['hedge_count'],
                weak_r['limitation_signals'], len(weak_r['generic_future_work'])]
strong_vals = [len(strong_r['overclaims']), strong_r['hedge_count'],
                strong_r['limitation_signals'], len(strong_r['generic_future_work'])]

# Normalize: for overclaims and generic future, 0 is best; for hedges/limitations, higher is better
def normalize_disc(vals):
    # criteria: overclaims (lower=better → invert), hedges (higher), limitations (higher), generic (lower → invert)
    oc, hd, lm, gf = vals
    return [max(0, 1 - oc/3), min(1, hd/4), min(1, lm/3), max(0, 1 - gf/2)]

weak_norm   = normalize_disc(weak_vals)
strong_norm = normalize_disc(strong_vals)

x = np.arange(len(criteria))
w = 0.35
ax.bar(x - w/2, weak_norm,   w, label='Weak',   color='tomato',    edgecolor='black', alpha=0.8)
ax.bar(x + w/2, strong_norm, w, label='Strong', color='steelblue', edgecolor='black', alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(criteria, fontsize=8)
ax.set_ylim(0, 1.2); ax.set_ylabel('Quality score (1.0 = ideal)')
ax.set_title('A. Discussion quality audit')
ax.legend(fontsize=9)
ax.axhline(1.0, color='grey', ls=':', lw=0.8)

# Panel B: Discussion structure diagram
ax = axes[1]
ax.axis('off')
struct = [
    ('D1', 'Core interpretation', 'What your result means (not restating it)', '#4e79a7'),
    ('D2', 'Comparison to prior work', 'How it relates to 3–5 key papers', '#f28e2b'),
    ('D3', 'Mechanism / explanation', 'Why it works — flag speculation', '#59a14f'),
    ('D4', 'Limitations', 'Specific: what the method cannot do', '#e15759'),
    ('D5', 'Future work', 'Specific, testable next steps only', '#76b7b2'),
]
for i, (code, title, desc, color) in enumerate(struct):
    y = 0.92 - i * 0.17
    ax.add_patch(mpatches.FancyBboxPatch((0.02, y - 0.07), 0.95, 0.14,
                                           boxstyle='round,pad=0.01', facecolor=color,
                                           alpha=0.25, transform=ax.transAxes))
    ax.text(0.06, y, f'{code}: {title}', transform=ax.transAxes,
              fontsize=9, fontweight='bold', va='center')
    ax.text(0.06, y - 0.045, desc, transform=ax.transAxes, fontsize=8,
              va='center', style='italic', color='#444')
ax.set_title('B. Discussion paragraph structure')

# Panel C: Overclaim spectrum
ax = axes[2]
ax.axis('off')
examples = [
    ('OVERCLAIMS', 'tomato', [
        '"Our model proves TF binding is\ndetermined by cooperativity."',
        '"This definitively solves the\nsequence specificity problem."',
        '"Results demonstrate deep learning\nalways outperforms PWMs."',
    ]),
    ('APPROPRIATE', 'steelblue', [
        '"Results are consistent with\ncooperativity contributing to..."',
        '"Our method reduces the AUROC\ngap on the ENCODE benchmark."',
        '"Gains suggest positional dependence\nis informative in these assays."',
    ]),
]
for col_idx, (label, color, items) in enumerate(examples):
    x_pos = col_idx * 0.5 + 0.01
    ax.text(x_pos + 0.22, 0.97, label, transform=ax.transAxes,
              fontsize=10, fontweight='bold', ha='center', color=color)
    for j, item in enumerate(items):
        y = 0.87 - j * 0.28
        ax.text(x_pos + 0.22, y, item, transform=ax.transAxes,
                  fontsize=8, ha='center', va='top',
                  bbox=dict(boxstyle='round,pad=0.3', facecolor=color, alpha=0.2))
ax.axvline(0.5, ymin=0.05, ymax=0.98, color='grey', ls='--', lw=1.5,
             transform=ax.transAxes)
ax.set_title('C. Overclaiming vs. appropriate hedging\n(same result, two framings)')

plt.suptitle('Module 18 NB06: Discussion and Conclusions', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('discussion_conclusions.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

See E06 in `exercises/README.md`: rewrite the weak discussion above,
removing all overclaims and adding one specific limitation paragraph.

---
## Step 10 — Quiz

1. What are the five standard discussion paragraphs and what does each contain?
2. Rewrite: "Our results prove that the LSTM is better than random forests for
   gene expression prediction." Using: RMSE 0.18 vs 0.23, n=50 genes, one dataset.
3. Write a one-sentence limitation for a study that trained a classifier on
   TCGA cancer data and evaluated it only on TCGA.
4. What is the difference between a specific future-work statement and a generic one?
   Rewrite: "Future work could explore more datasets" to be specific.

---
## Step 12 — Reflection

> *[For MP01, write D4 (limitations paragraph) for your chosen result. Include:
> the specific limitation, what type of data or condition it does not cover,
> and the biological or technical reason it matters. Target: 50–80 words.]*

---
*Next: `07_publication_quality_figures.ipynb`*